In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.python.keras import backend
from keras.engine.functional import Functional
from keras import layers

In [ ]:
# Inputs
path_to_neqopnn = '../Neq-DeepONet.final/delta/saving/neq_deeponet_infer_n250/SavedModel/'
path_to_cgopnn = '../CG-DeepONet.final/merge/run.03/saving/cg_deeponet_infer_n250/SavedModel'
dim = {
    "inp": [29, 1, 1],
    "out": [1, 27, 1]
}

In [ ]:
# Load each models
neqopnn = tf.keras.models.load_model(path_to_neqopnn)
cgopnn = tf.keras.models.load_model(path_to_cgopnn)

In [ ]:
neq = [w.name for w in neqopnn.weights]

In [ ]:
cg = [w.name for w in cgopnn.weights]

In [ ]:
len(cg)+ len(neq)

In [ ]:
len(np.unique(tf.nest.flatten([cg,neq])))

In [ ]:
# Function call
def call(x, dim):
    # Input manipulation
    x = tf.split(x, dim["inp"], axis=1)
    y, tol = x[:-1], x[-1]
    # Kernel - Neq-DeepONet
    mask = neqopnn(x)
    # Kernel - CG-DeepONet
    y = tf.nest.flatten([y, mask])
    y = cgopnn(y)
    # Output manipulation
    y = layers.Concatenate()(y)
    return y

In [ ]:
# Input shape
n = neqopnn.inputs[-1].shape[0]
d = np.sum([i.shape[1] for i in neqopnn.inputs])
# Inputs/Outputs
inputs = tf.keras.Input(tensor=tf.zeros((n,d), dtype=backend.floatx()))
outputs = call(inputs, dim)
# Build model
net = Functional(inputs, outputs, name="CGNeqNO")

In [ ]:
# Save model
net.save(
    'SavedModel',
    overwrite=True,
    include_optimizer=False,
    save_format='tf',
    signatures=None,
    options=None,
    save_traces=True
)